# WildlifeVision YOLO11 Trainer

Train a bird detector from the reviewed dataset exported by WildlifeVision, then export the ONNX model and class labels for the browser app.

**Workflow:** import and label images in WildlifeVision, prepare and verify the bounding boxes, save the YOLO dataset ZIP, then open this notebook with **Train in Colab**.

In [ ]:
!git clone https://github.com/EdebN/curly-system.git
%cd /content/curly-system
!pip install -q -r training/requirements.txt

## 1. Select the T4 runtime

Choose **Runtime → Change runtime type → T4 GPU** and save before training.

## 2. Upload your dataset and (optionally) your previous checkpoint

Upload the `WildlifeVision-YOLO-dataset.zip` exported from Bird Library. To continue improving an existing detector, select its `best.pt` file in the same upload dialog. If you do not upload a checkpoint, training starts from `yolo11n.pt`.

**Important:** ONNX files are inference exports and cannot be fine-tuned. Keep the `.pt` checkpoint. For new species, include the previous species' labelled images in the new dataset as well, so the detector does not simply forget them.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
dataset_files = [name for name in uploaded if name.lower().endswith('.zip')]
checkpoint_files = [name for name in uploaded if name.lower().endswith('.pt')]
if len(dataset_files) != 1:
    raise ValueError(f'Upload exactly one dataset ZIP. Found: {dataset_files}')
if len(checkpoint_files) > 1:
    raise ValueError(f'Upload at most one .pt checkpoint. Found: {checkpoint_files}')
dataset_zip = str(Path(dataset_files[0]).resolve())
model_checkpoint = str(Path(checkpoint_files[0]).resolve()) if checkpoint_files else 'yolo11n.pt'
print('Dataset:', dataset_zip)
print('Starting checkpoint:', model_checkpoint)
print('Using previous trained weights.' if checkpoint_files else 'Starting from the standard YOLO11n checkpoint.')

## 3. Add optional COCO background examples

COCO's `train2014.zip` is the full 83K-image archive (about 13 GB), not a background-only collection. This optional step instead downloads the 2014 instance-annotation index, selects a modest sample of images with **no annotated `bird` instances**, and downloads only those individual photos. It adds them with empty YOLO label files to the training and validation splits. The full 13 GB image archive is not needed.

**Caution:** these are images with no COCO bird annotation, not guaranteed bird-free images. COCO annotations can miss small or incidental birds. Keep the sample moderate and inspect the resulting model for false negatives as well as false positives. Set `ADD_COCO_BACKGROUND_IMAGES = False` to skip this step.

In [ ]:
import json, random, requests, zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from pathlib import Path
from PIL import Image

ADD_COCO_BACKGROUND_IMAGES = True
COCO_BACKGROUND_COUNT = 400  # modest starting sample; increase only after checking results
COCO_ANNOTATIONS_URL = 'https://images.cocodataset.org/annotations/annotations_trainval2014.zip'

def add_coco_background_examples(dataset_zip, count=400):
    output_root = Path('/content/wildlifevision-training')
    dataset_root = output_root / 'dataset'
    if not (dataset_root / 'dataset.yaml').is_file():
        import shutil
        if dataset_root.exists():
            shutil.rmtree(dataset_root)
        dataset_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(dataset_zip) as archive:
            archive.extractall(dataset_root)
        if not (dataset_root / 'dataset.yaml').is_file():
            nested = list(dataset_root.rglob('dataset.yaml'))
            if len(nested) != 1:
                raise FileNotFoundError('Could not locate exactly one dataset.yaml in the uploaded ZIP.')
            dataset_root = nested[0].parent

    annotation_zip = Path('/content/annotations_trainval2014.zip')
    if not annotation_zip.is_file():
        print('Downloading COCO 2014 annotation index (about 241 MB)…')
        response = requests.get(COCO_ANNOTATIONS_URL, timeout=180)
        response.raise_for_status()
        annotation_zip.write_bytes(response.content)
    with zipfile.ZipFile(annotation_zip) as archive:
        member = next((name for name in archive.namelist() if name.endswith('instances_train2014.json')), None)
        if not member:
            raise FileNotFoundError('instances_train2014.json was not found in the COCO annotations archive.')
        coco = json.loads(archive.read(member))

    bird_category_ids = {item['id'] for item in coco['categories'] if item['name'].strip().lower() == 'bird'}
    if not bird_category_ids:
        raise ValueError('COCO annotation metadata did not contain the bird category.')
    images_with_annotated_birds = {
        annotation['image_id'] for annotation in coco['annotations']
        if annotation['category_id'] in bird_category_ids
    }
    candidates = [item for item in coco['images'] if item['id'] not in images_with_annotated_birds]
    random.Random(20261031).shuffle(candidates)
    selected = candidates[:count]
    val_count = max(1, round(len(selected) * 0.1)) if len(selected) > 1 else 0
    validation_ids = {item['id'] for item in selected[:val_count]}
    print(f'Found {len(candidates):,} images without an annotated COCO bird; attempting {len(selected)} downloads.')

    def fetch_one(item):
        url = item.get('coco_url') or f"https://images.cocodataset.org/train2014/{item['file_name']}"
        url = url.replace('http://', 'https://', 1)
        response = requests.get(url, timeout=45)
        response.raise_for_status()
        payload = response.content
        with Image.open(BytesIO(payload)) as image:
            image.verify()
        split = 'val' if item['id'] in validation_ids else 'train'
        image_dir = dataset_root / 'images' / split
        label_dir = dataset_root / 'labels' / split
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)
        filename = Path(item['file_name']).name
        (image_dir / filename).write_bytes(payload)
        (label_dir / (Path(filename).stem + '.txt')).write_text('', encoding='utf-8')
        return split, filename

    added = {'train': 0, 'val': 0}
    failures = []
    with ThreadPoolExecutor(max_workers=8) as pool:
        futures = {pool.submit(fetch_one, item): item for item in selected}
        for future in as_completed(futures):
            try:
                split, _ = future.result()
                added[split] += 1
            except Exception as error:
                failures.append((futures[future]['file_name'], str(error)))
    print(f"Added COCO negatives: {added['train']} train, {added['val']} validation; {len(failures)} download failures.")
    if failures:
        print('First few download failures:', failures[:5])
    print('Dataset folder ready:', dataset_root)
    return dataset_root

if ADD_COCO_BACKGROUND_IMAGES:
    dataset_root = add_coco_background_examples(dataset_zip, COCO_BACKGROUND_COUNT)
else:
    print('Skipping optional COCO background examples.')

## 3. Train YOLO11 and export ONNX

Run this cell to display the training button, then click it. If you supplied `best.pt`, the trainer starts from those weights. With the same species list, it fine-tunes the detector on the expanded dataset. If you added species, Ultralytics transfers compatible weights and adapts the detection head to the new class list.

In [ ]:
import subprocess, sys, torch, ipywidgets as widgets
from IPython.display import display

train_button = widgets.Button(description='Train YOLO11', button_style='success', icon='play', tooltip='Fine-tune and export the detector')
train_output = widgets.Output(layout={'border': '1px solid #888', 'max_height': '420px', 'overflow_y': 'auto'})

def start_yolo_training(_):
    train_button.disabled = True
    train_button.description = 'Training…'
    with train_output:
        print('Preparing YOLO11 training…')
        device = '0' if torch.cuda.is_available() else 'cpu'
        print('Training device:', device)
        training_dataset = str(dataset_root) if 'dataset_root' in globals() and (Path(dataset_root) / 'dataset.yaml').is_file() else dataset_zip
        print('Training dataset:', training_dataset)
        command = [
            sys.executable, 'training/train_yolo.py', training_dataset,
            '--model', model_checkpoint,
            '--epochs', '100', '--imgsz', '640', '--batch', '-1',
            '--workers', '2', '--device', device,
            '--output', '/content/wildlifevision-training',
            '--name', 'bird-detector'
        ]
        try:
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end='')
            exit_code = process.wait()
            if exit_code != 0:
                raise subprocess.CalledProcessError(exit_code, command)
            print('\nTraining and ONNX export completed. Download the new ONNX, labels, and checkpoint below.')
            train_button.description = 'Training complete'
            train_button.button_style = 'success'
        except Exception as error:
            print('\nTraining failed:', error)
            train_button.description = 'Retry training'
            train_button.button_style = 'danger'
        finally:
            train_button.disabled = False

train_button.on_click(start_yolo_training)
display(train_button, train_output)

## 4. Download the updated detector

Download all three files. Use the new `best.onnx` and `best.labels.json` in WildlifeVision. Keep the new `best.pt` as the checkpoint for the next training round.

In [ ]:
from google.colab import files
from pathlib import Path

weights = Path('/content/wildlifevision-training/runs/bird-detector/weights')
for filename in ['best.onnx', 'best.labels.json', 'best.pt']:
    path = weights / filename
    if not path.is_file():
        raise FileNotFoundError(f'Expected training output is missing: {path}')
    print('Downloading', filename)
    files.download(str(path))

## 5. Load the updated model in WildlifeVision

In **Live camera**, choose the new `best.onnx`, upload the matching `best.labels.json`, then select **Load YOLO11**. The labels file must always match the ONNX model.

For the next update, repeat the process with the latest `best.pt` and a dataset ZIP containing all species you want the detector to retain.